In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


DAYLIGHT_THRESHOLD = 0.05
DATASET_DIR = r"G:\Projects\Depi\Depi Grid\Datasets\unisolar"  

In [2]:
def load_and_prep_anomaly_data(base_dir):
    gen_path = os.path.join(base_dir, "Plant_1_Generation_Data.csv")
    weather_path = os.path.join(base_dir, "Plant_1_Weather_Sensor_Data.csv")
    
    df_gen = pd.read_csv(gen_path)
    df_weather = pd.read_csv(weather_path)

    df_gen['DATE_TIME'] = pd.to_datetime(df_gen['DATE_TIME'], format='%d-%m-%Y %H:%M')
    df_weather['DATE_TIME'] = pd.to_datetime(df_weather['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')

    df_merged = pd.merge(df_gen, df_weather[['DATE_TIME', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']], 
                         on='DATE_TIME', how='inner')
    
    df_daylight = df_merged[df_merged['IRRADIATION'] > DAYLIGHT_THRESHOLD].copy()
    df_daylight = df_daylight.sort_values(['SOURCE_KEY', 'DATE_TIME']).reset_index(drop=True)
    
    return df_daylight

In [3]:
class TheoreticalYieldModel:
    def __init__(self):
        self.model = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, random_state=42)
        self.features = ['IRRADIATION', 'MODULE_TEMPERATURE', 'AMBIENT_TEMPERATURE']
        self.target = 'DC_POWER'
        
    def train(self, df):
        X = df[self.features]
        y = df[self.target]
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        self.model.fit(X_train, y_train)
        predictions = self.model.predict(X_test)
        
        r2 = r2_score(y_test, predictions)
        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        target_mean = y_test.mean()
        
        print("="*60)
        print("      THEORETICAL YIELD REGRESSOR METRICS (BASELINE)")
        print("="*60)
        print(f" R² Score:                       {r2:.4f}")
        print(f" Mean Absolute Error (MAE):      {mae:.2f} kW ({ (mae/target_mean)*100:.2f}% of mean)")
        print(f" Root Mean Squared Error (RMSE): {rmse:.2f} kW")
        print("="*60)
        
        return {"r2": r2, "mae": mae, "rmse": rmse}
        
    def predict_expected_yield(self, df):
        return self.model.predict(df[self.features])

In [4]:
class SoilingAnomalyDetector:
    def __init__(self, deficit_threshold_pct=0.15, rolling_window_hours=72):
        self.threshold = deficit_threshold_pct
        self.window = rolling_window_hours * 4   # each interval is 15 minutes
        
    def flag_anomalies(self, df):
        df['POWER_DEFICIT'] = df['EXPECTED_DC_POWER'] - df['DC_POWER']
        df['EFFICIENCY_LOSS'] = df['POWER_DEFICIT'] / (df['EXPECTED_DC_POWER'] + 1e-5)
        
        results = []
        for inverter_id, group in df.groupby('SOURCE_KEY'):
            group = group.copy()
            group['SUSTAINED_LOSS'] = group['EFFICIENCY_LOSS'].rolling(window=self.window, min_periods=self.window//2).mean()
            group['SOILING_ANOMALY'] = (group['SUSTAINED_LOSS'] > self.threshold).astype(int)
            results.append(group)
            
        return pd.concat(results).sort_index()


def evaluate_anomaly_impact(df_evaluated, threshold_pct, window_hours):
    anomalies = df_evaluated[df_evaluated['SOILING_ANOMALY'] == 1]
    
    total_intervals = len(df_evaluated)
    anomaly_intervals = len(anomalies)
    degraded_time_pct = (anomaly_intervals / total_intervals) * 100 if total_intervals > 0 else 0
    
    print("="*60)
    print("          SOILING & DEGRADATION OPERATIONAL REPORT")
    print("="*60)
    print(f" Trigger Parameters: > {threshold_pct*100}% loss sustained for {window_hours} hrs")
    print("-" * 60)
    print(f" Total Daylight Intervals:       {total_intervals}")
    print(f" Flagged Anomaly Intervals:      {anomaly_intervals}")
    print(f" System Time in Degraded State:  {degraded_time_pct:.2f}%")
    
    if anomaly_intervals > 0:
        avg_loss_pct = anomalies['EFFICIENCY_LOSS'].mean() * 100
        max_loss_pct = anomalies['EFFICIENCY_LOSS'].max() * 100
        total_kw_lost = anomalies['POWER_DEFICIT'].sum()
        
        print("-" * 60)
        print(f" Average Efficiency Drop:        {avg_loss_pct:.2f}%")
        print(f" Peak Efficiency Drop:           {max_loss_pct:.2f}%")
        print(f" Total Estimated Power Lost:     {total_kw_lost:,.2f} kW")
        
        worst_inverter = anomalies.groupby('SOURCE_KEY')['POWER_DEFICIT'].sum().idxmax()
        worst_inverter_loss = anomalies.groupby('SOURCE_KEY')['POWER_DEFICIT'].sum().max()
        print(f"\n Critical Action: Inverter '{worst_inverter}' requires ")
        print(f" cleaning/inspection. (Contributed {worst_inverter_loss:,.2f} kW to total loss)")
    print("="*60)

In [ ]:
df_anomaly = load_and_prep_anomaly_data(DATASET_DIR)

yield_model = TheoreticalYieldModel()
metrics = yield_model.train(df_anomaly)

df_anomaly['EXPECTED_DC_POWER'] = yield_model.predict_expected_yield(df_anomaly)

detector = SoilingAnomalyDetector(deficit_threshold_pct=0.15, rolling_window_hours=72)
df_evaluated = detector.flag_anomalies(df_anomaly)

evaluate_anomaly_impact(df_evaluated, detector.threshold, 72)

joblib.dump(yield_model.model, "solar_theoretical_yield_model.pkl")

anomaly_config = {
    "features": yield_model.features,
    "target": yield_model.target,
    "daylight_threshold_irradiation": DAYLIGHT_THRESHOLD,
    "anomaly_deficit_threshold_pct": detector.threshold,
    "rolling_window_intervals": detector.window
}

with open("soiling_anomaly_config.json", "w") as f:
    json.dump(anomaly_config, f)

print("\nModel and configuration saved successfully.")

      THEORETICAL YIELD REGRESSOR METRICS (BASELINE)
 R² Score:                       0.9617
 Mean Absolute Error (MAE):      362.42 kW (5.59% of mean)
 Root Mean Squared Error (RMSE): 690.05 kW
          SOILING & DEGRADATION OPERATIONAL REPORT
 Trigger Parameters: > 15.0% loss sustained for 72 hrs
------------------------------------------------------------
 Total Daylight Intervals:       33266
 Flagged Anomaly Intervals:      0
 System Time in Degraded State:  0.00%

Model and configuration saved successfully.
